# KNN-MLP Results Plotter

This notebook fetches multiple Weights & Biases runs matching flexible config filters (including list-valued options), aggregates the selected metric across seeds, and plots mean with optional ranges (min/max, std, stderr). You can save figures as PDF/PNG. Edit the configuration cell to change filters, metric, and styling.

In [ ]:
# Imports
import os

import numpy as np
import matplotlib.pyplot as plt
import wandb

In [ ]:
# Helper import: functions moved to wandb_plot_utils.py for readability
from wandb_plot_utils import (
    plot_metric_over_epochs,
    save_current_figure,
    update_and_get_config_options,
    print_config_options,
    build_plot_filename,
)

# Project selection and available config options (run this first)
ENTITY = "haraghi"
PROJECT = "knn-mlp-regression-relative-multiseed"
OPTIONS_CACHE_DIR = "cache"
CONFIG_KEYS_TO_SCAN = None    # e.g., ["lr", "k", "feature_type"] or None for all keys
UPDATE_CONFIG_OPTIONS = False # set True to refresh from server
MAX_RUNS_TO_SCAN = None       # optional limit for speed

try:
    api = wandb.Api()
    options = update_and_get_config_options(
        api=api,
        entity=ENTITY,
        project=PROJECT,
        cache_dir=OPTIONS_CACHE_DIR,
        force_refresh=UPDATE_CONFIG_OPTIONS,
        config_keys=CONFIG_KEYS_TO_SCAN,
        max_runs=MAX_RUNS_TO_SCAN,
    )
    print("Available config options with multiple values (from cache unless refreshed):")
    print_config_options(options, min_values=2)
except Exception as e:
    print(f"Error while retrieving config options: {e}")

# Tip: during development you can reload the module without restarting the kernel:
# import importlib, wandb_plot_utils as wpu
# importlib.reload(wpu)
# from wpu import plot_metric_over_epochs, save_current_figure, update_and_get_config_options, print_config_options, build_plot_filename

In [ ]:
# Configuration: choose filters and plotting options

# Filters can be scalars or lists. All list entries will be combined (cartesian product)
CONFIG_FILTER = {
    # examples (edit to your needs):
    "test_train_split": ["temporal"],  # e.g., ["temporal", "random"]
    "lr": [1e-4],                       # e.g., [1e-4, 5e-5]
    "k": [10, 50],                      # e.g., [10, 50, 100]
    "feature_type": ["both","original"],  # e.g., ["both", "mlp_only", "knn_only"]
    "max_epochs": 500,
}

# Plot controls
METRIC = "val_loss"              # metric key in W&B history, e.g., "val_loss"
RANGE_TYPE = "std"               # "minmax" | "std" | "stderr" | "none"
INCLUDE_FILL = True
ALPHA = 0.25                     # transparency for fill_between
LINEWIDTH = 2.0
FIGSIZE = (8, 5)
GRID = True
SHOW_LEGEND = True

# Color control for consistent mapping by config key (e.g., feature_type)
COLOR_BY_KEY = "feature_type"    # set to None to disable color mapping by key

# Caching controls
USE_CACHE = True                 # if True, load from cache when available
REFRESH_CACHE = False            # if True, refetch from W&B and overwrite cache
CACHE_DIR = "cache"

# Saving controls
SAVE_DIR = "outputs"  # where to save figures
SAVE_NAME = None                     # optional override; if None, name is auto-built from config
SAVE_PDF = True
SAVE_PNG = False
ADD_TIMESTAMP = False               # do not add time; filenames are configuration-specific

In [ ]:
# Execute: fetch, plot, and save
try:
    api = wandb.Api()
    title = f"{METRIC} vs Epoch"
    plot_metric_over_epochs(
        api=api,
        entity=ENTITY,
        project=PROJECT,
        base_filter=CONFIG_FILTER,
        metric=METRIC,
        range_type=RANGE_TYPE,
        include_fill=INCLUDE_FILL,
        alpha=ALPHA,
        linewidth=LINEWIDTH,
        figsize=FIGSIZE,
        title=title,
        grid=GRID,
        show_legend=SHOW_LEGEND,
        color_by_key=COLOR_BY_KEY,
        use_cache=USE_CACHE,
        refresh_cache=REFRESH_CACHE,
        cache_dir=CACHE_DIR,
    )

    # Build a configuration-specific filename if not provided
    auto_name = build_plot_filename(
        base_filter=CONFIG_FILTER,
        metric=METRIC,
        range_type=RANGE_TYPE,
        color_by_key=COLOR_BY_KEY,
    )
    selected_name = SAVE_NAME if SAVE_NAME else auto_name

    if SAVE_PNG or SAVE_PDF:
        save_current_figure(
            save_dir=SAVE_DIR,
            save_name=selected_name,
            save_png=SAVE_PNG,
            save_pdf=SAVE_PDF,
            timestamp=ADD_TIMESTAMP,
        )
        print(f"Saved figure(s) as '{selected_name}.pdf/png' to {os.path.abspath(SAVE_DIR)}")

    plt.show()
except Exception as e:
    print(f"Error during plotting: {e}")

## Notes

- To compare multiple settings on the same plot, set list-valued entries in `CONFIG_FILTER` (e.g., `"k": [10, 50, 100]`, `"lr": [1e-4, 5e-5]`). All combinations are plotted with separate lines.
- Choose `RANGE_TYPE` among `"minmax"`, `"std"`, `"stderr"`, or `"none"` for the shaded area around the mean. Toggle shading with `INCLUDE_FILL`.
- Set `SAVE_PDF=True` to export vector graphics; `SAVE_PNG=True` for raster. Files save to `SAVE_DIR` with the base `SAVE_NAME`.
- If you see "No runs" messages, adjust `CONFIG_FILTER` to match your W&B runs (entity/project must be correct).
- You can change `METRIC` to any history key logged in your runs (e.g., `"train_loss"`).

## Batch generation: 6 figures for hidden_dim × k

This section generates one figure per combination of hidden_dim and k ∈ {5, 10, 50}. Each figure plots feature_type ∈ [original, eig, filter, both] with consistent colors based on feature_type.

In [ ]:
# Batch: generate 6 figures for combinations of hidden_dim and k


# Adjust the hidden dimensions here (two values recommended for 6 figures with k=[5,10,50])
HIDDEN_DIMS = [64, 128]  # change as needed
K_LIST = [5, 10, 50]  # change as needed
FEATURE_TYPES = ["original", "eig", "filter", "both"]


try:
    api = wandb.Api()
    for hd in HIDDEN_DIMS:
        for k in K_LIST:
            base_filter = {
                "test_train_split": "temporal",
                "lr": 1e-4,
                "max_epochs": 200,
                "hidden_dim": hd,
                "k": k,
                # Plot all feature types in this figure
                "feature_type": FEATURE_TYPES,
            }


            # Plot figure
            plot_metric_over_epochs(
                api=api,
                entity=ENTITY,
                project=PROJECT,
                base_filter=base_filter,
                metric=METRIC,
                range_type=RANGE_TYPE,
                include_fill=INCLUDE_FILL,
                alpha=ALPHA,
                linewidth=LINEWIDTH,
                figsize=FIGSIZE,
                title=f"{METRIC} vs Epoch | hidden_dim={hd}, k={k}",
                grid=GRID,
                show_legend=SHOW_LEGEND,
                color_by_key="feature_type",
                use_cache=USE_CACHE,
                refresh_cache=REFRESH_CACHE,
                cache_dir=CACHE_DIR,
            )


            # Auto filename from config (no timestamp)
            auto_name = build_plot_filename(
                base_filter=base_filter,
                metric=METRIC,
                range_type=RANGE_TYPE,
                color_by_key="feature_type",
                prefix="batch",
            )
            if SAVE_PDF or SAVE_PNG:
                save_current_figure(
                    save_dir=SAVE_DIR,
                    save_name=auto_name,
                    save_png=SAVE_PNG,
                    save_pdf=SAVE_PDF,
                    timestamp=False,
                )
                print(f"Saved: {auto_name} (.pdf/.png) -> {os.path.abspath(SAVE_DIR)}")


            plt.show()
            plt.close()
except Exception as e:
    print(f"Batch generation error: {e}")

## Specific figure: hidden_dim=128, k=50, lr=1e-4, max_epochs=500, temporal split

This cell plots feature_type ∈ [original, original_random_augmented, original_repeated_augmented, eig, filter, both] with consistent colors by feature_type and saves a configuration-specific filename (no timestamp).

In [ ]:
# One-off figure: hidden_dim=128, k=50 with six feature types
FEATURE_TYPES_SPECIFIC = [
    "original",
    "original_random_augmented",
    "original_repeated_augmented",
    "eig",
    "filter",
    "both",
]

try:
    api = wandb.Api()
    base_filter = {
        "test_train_split": "temporal",
        "lr": 1e-4,
        "max_epochs": 200,
        "hidden_dim": 128,
        "k": 50,
        "feature_type": FEATURE_TYPES_SPECIFIC,
    }

    plot_metric_over_epochs(
        api=api,
        entity=ENTITY,
        project=PROJECT,
        base_filter=base_filter,
        metric=METRIC,
        range_type=RANGE_TYPE,
        include_fill=INCLUDE_FILL,
        alpha=ALPHA,
        linewidth=LINEWIDTH,
        figsize=FIGSIZE,
        title=f"{METRIC} vs Epoch | hidden_dim=128, k=50",
        grid=GRID,
        show_legend=SHOW_LEGEND,
        color_by_key="feature_type",
        use_cache=USE_CACHE,
        refresh_cache=REFRESH_CACHE,
        cache_dir=CACHE_DIR,
    )

    auto_name = build_plot_filename(
        base_filter=base_filter,
        metric=METRIC,
        range_type=RANGE_TYPE,
        color_by_key="feature_type",
        prefix="specific",
    )
    selected_name = SAVE_NAME if SAVE_NAME else auto_name

    if SAVE_PNG or SAVE_PDF:
        save_current_figure(
            save_dir=SAVE_DIR,
            save_name=selected_name,
            save_png=SAVE_PNG,
            save_pdf=SAVE_PDF,
            timestamp=False,
        )
        print(f"Saved figure(s) as '{selected_name}.pdf/png' to {os.path.abspath(SAVE_DIR)}")

    plt.show()
except Exception as e:
    print(f"Error during specific figure plotting: {e}")

## Specific figure: hidden_dim=128, k=50, lr=1e-4, max_epochs=200, temporal split (exclude_xy variants)

This cell plots feature_type ∈ [original, both, both_exclude_xy, eig_exclude_xy, filter_exclude_xy] with consistent colors by feature_type and saves a configuration-specific filename (no timestamp).

In [ ]:
# One-off figure: hidden_dim=128, k=50 with exclude_xy variants
FEATURE_TYPES_EXCLUDE_XY = [
    "original",
    "both",
    "both_exclude_xy",
    "eig_exclude_xy",
    "filter_exclude_xy",
]

try:
    api = wandb.Api()
    base_filter = {
        "test_train_split": "temporal",
        "lr": 1e-4,
        "max_epochs": 200,
        "hidden_dim": 128,
        "k": 50,
        "feature_type": FEATURE_TYPES_EXCLUDE_XY,
    }

    plot_metric_over_epochs(
        api=api,
        entity=ENTITY,
        project=PROJECT,
        base_filter=base_filter,
        metric=METRIC,
        range_type=RANGE_TYPE,
        include_fill=INCLUDE_FILL,
        alpha=ALPHA,
        linewidth=LINEWIDTH,
        figsize=FIGSIZE,
        title=f"{METRIC} vs Epoch | hidden_dim=128, k=50 (exclude_xy)",
        grid=GRID,
        show_legend=SHOW_LEGEND,
        color_by_key="feature_type",
        use_cache=USE_CACHE,
        refresh_cache=REFRESH_CACHE,
        cache_dir=CACHE_DIR,
    )

    auto_name = build_plot_filename(
        base_filter=base_filter,
        metric=METRIC,
        range_type=RANGE_TYPE,
        color_by_key="feature_type",
        prefix="specific_exclude_xy",
    )
    selected_name = SAVE_NAME if SAVE_NAME else auto_name

    if SAVE_PNG or SAVE_PDF:
        save_current_figure(
            save_dir=SAVE_DIR,
            save_name=selected_name,
            save_png=SAVE_PNG,
            save_pdf=SAVE_PDF,
            timestamp=False,
        )
        print(f"Saved figure(s) as '{selected_name}.pdf/png' to {os.path.abspath(SAVE_DIR)}")

    plt.show()
except Exception as e:
    print(f"Error during specific exclude_xy plotting: {e}")

In [ ]:
# Generate time-augmentation figures for hidden_dim=128, lr=1e-4, max_epochs=500
FEATURE_TYPES_TIMEAUG = ["original", "filter", "eig", "both", "original_time_augmented"]
K_LIST_TIMEAUG = [5, 10, 50]

try:
    api = wandb.Api()
    for k_val in K_LIST_TIMEAUG:
        base_filter = {
            "test_train_split": "temporal",
            "lr": 1e-4,
            "max_epochs": 500,
            "hidden_dim": 128,
            "k": k_val,
            "feature_type": FEATURE_TYPES_TIMEAUG,
        }

        # Plot figure for this k
        plot_metric_over_epochs(
            api=api,
            entity=ENTITY,
            project=PROJECT,
            base_filter=base_filter,
            metric=METRIC,
            range_type=RANGE_TYPE,
            include_fill=INCLUDE_FILL,
            alpha=ALPHA,
            linewidth=LINEWIDTH,
            figsize=FIGSIZE,
            title=f"{METRIC} vs Epoch | hidden_dim=128, k={k_val} (time-aug)",
            grid=GRID,
            show_legend=SHOW_LEGEND,
            color_by_key="feature_type",
            use_cache=USE_CACHE,
            refresh_cache=REFRESH_CACHE,
            cache_dir=CACHE_DIR,
        )

        # Save using deterministic name
        auto_name = build_plot_filename(
            base_filter=base_filter,
            metric=METRIC,
            range_type=RANGE_TYPE,
            color_by_key="feature_type",
            prefix=f"specific_timeaug_k{k_val}",
        )
        selected_name = SAVE_NAME if SAVE_NAME else auto_name

        if SAVE_PNG or SAVE_PDF:
            save_current_figure(
                save_dir=SAVE_DIR,
                save_name=selected_name,
                save_png=SAVE_PNG,
                save_pdf=SAVE_PDF,
                timestamp=False,
            )
            print(f"Saved: {selected_name} -> {os.path.abspath(SAVE_DIR)}")

        plt.show()
        plt.close()
except Exception as e:
    print(f"Error generating time-aug figures: {e}")